In [1]:
import findspark
import os
#findspark.init() 
SPARK_HOME='/opt/cloudera/parcels/CDH/lib/spark'
# SPARK_HOME='/home/qiany/.conda/envs/py37'
# os.environ['SPARK_HOME'] = '/home/qiany/.conda/envs/py37'
findspark.init(SPARK_HOME)

In [2]:
import time
import math
import copy
import csv
import json
import os
import codecs
import subprocess
#from hdfs import InsecureClient
import numpy as np
#from pyspark import SparkContext
from pyspark import SQLContext
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.functions import expr, create_map, array_union,flatten,array_sort,coalesce,broadcast,collect_list, collect_set, udf, array_remove, log, lit, first, col, array, sort_array,split, explode, desc, asc, row_number,isnan, when, count
from pyspark.sql.types import *
import rtree
from pyspark.sql import Window
#import igraph
#from igraph import Graph
import geofeather
from pyspark.storagelevel import StorageLevel

In [3]:
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_small_crop_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_3_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_small_crop_tiny_test_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_3_crop_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T183726_185444_1/ALS_L1B_20190410T183726_185444_1.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T153919_155000_1/ALS_L1B_20190410T153919_155000_1.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T174554_181213/ALS_L1B_20190410T174554_181213_3_as_100000.off

t_whole_program_0 = time.time()

tin_file = input("Here is a programe to compute the Forman gradient, please input the absolute or relative path in local disk to your TIN file:")
# tin_file = '/local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T183726_185444_1/ALS_L1B_20190410T183726_185444_1.off'

print("\n********************")

# get the directory to the TIN file
tin_directory = os.path.dirname(tin_file)
print("tin_directory: ", tin_directory)

# seaice
# seaice_11122025
# seaice_simplify_ele
# ALS_L1B_20190410T174554_181213_small_crop_tiny_test
# ALS_L1B_20190410T174554_181213_small_crop_tiny_test_pers_0.5
# ALS_L1B_20190410T174554_181213_3_crop_pers_0.25
# ALS_L1B_20190410T174554_181213_3_pers_025
# ALS_L1B_20190410T174554_181213_3_pers_050
# ALS_L1B_20190410T174554_181213_3_pers_010
# ALS_L1B_20190410T174554_181213_3_pers_020
# ALS_L1B_20190410T174554_181213_3_pers_040
# ALS_L1B_20190410T174554_181213_3_pers_030
# ALS_L1B_20190410T183726_185444_1_pers_025
# ALS_L1B_20190410T153919_155000_1_pers_025
# ALS_L1B_20190410T174554_181213_pers_025
# ALS_L1B_20190410T174554_181213_3_pers_015
# ALS_L1B_20190410T174554_181213_3_pers_035
# ALS_L1B_20190410T174554_181213_3_pers_045

# ALS_L1B_20190410T153919_155000_1_pers_020
# ALS_L1B_20190410T153919_155000_1_pers_015
# ALS_L1B_20190410T153919_155000_1_pers_010
# ALS_L1B_20190410T153919_155000_1_pers_030
# ALS_L1B_20190410T153919_155000_1_pers_035
# ALS_L1B_20190410T153919_155000_1_pers_040
# ALS_L1B_20190410T153919_155000_1_pers_045
# ALS_L1B_20190410T153919_155000_1_pers_050

# ALS_L1B_20190410T183726_185444_1_pers_020
# ALS_L1B_20190410T183726_185444_1_pers_015
# ALS_L1B_20190410T183726_185444_1_pers_010
# ALS_L1B_20190410T183726_185444_1_pers_030
# ALS_L1B_20190410T183726_185444_1_pers_035
# ALS_L1B_20190410T183726_185444_1_pers_040
# ALS_L1B_20190410T183726_185444_1_pers_045
# ALS_L1B_20190410T183726_185444_1_pers_050


directory = input("The HDFS directory where you want to store DataFrames:")
# directory = 'ALS_L1B_20190410T183726_185444_1_pers_020'

# simplify_with_order = input("Simplify with order('y' or 'n'):") or "n"
simplify_with_order = "n"

porsist_thre = input("What is the persistence value threshold (m) you want to set:") or "0.25"
# porsist_thre = '0.20'

# already_resimplify_idx = input("How many resimplification have you already performed:")
already_resimplify_idx = '1'

# already_merge_idx = input("How many merging processes have you already performed:")
already_merge_idx = '2'

round_idx = input("which round of new resimplification you want to perform:")
# round_idx = '1'

# get the basename to the TIN file
tin_basename = os.path.basename(tin_file) # input_vertices_2.off
print("tin_basename: ", tin_basename)

# get the filename of the TIN file
tin_filename = os.path.splitext(tin_basename)[0] # input_vertices_2
print("tin_filename: ", tin_filename)

# get the type of TIN file: off, tri, etc
tin_extension = os.path.splitext(tin_basename)[1] # .off
print("tin_extension: ", tin_extension)
print("\n********************")

Here is a programe to compute the Forman gradient, please input the absolute or relative path in local disk to your TIN file: /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T153919_155000_1/ALS_L1B_20190410T153919_155000_1.off



********************
tin_directory:  /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T153919_155000_1


The HDFS directory where you want to store DataFrames: ALS_L1B_20190410T153919_155000_1_pers_020
What is the persistence value threshold (m) you want to set: 0.20
which round of new resimplification you want to perform: 6


tin_basename:  ALS_L1B_20190410T153919_155000_1.off
tin_filename:  ALS_L1B_20190410T153919_155000_1
tin_extension:  .off

********************


In [4]:
# filtra is the order of each vertex, the order is obtained by ranking elevation values of vertices
# filtra = input("Do you have filtration data?") or "yes"
filtra = 'yes'

if filtra.lower() == 'no':
    Basic_Data = input("Do you have basic pts and tri data?")
    
Num_executor = input("spark.executor.instances:") or "32"
Num_core_per_executor = input("spark.executor.cores:") or "3"
Memory_executor = input("spark.executor.memory? Please end with 'g':") or "56g"
MemoryOverhead_executor = input("spark.executor.memoryOverhead? Please end with 'g':") or "24g"

# Num_core_per_driver = Num_core_per_executor
# Memory_driver = Memory_executor
# MemoryOverhead_driver = MemoryOverhead_executor

Num_core_per_driver = '5'
Memory_driver = '56g'
MemoryOverhead_driver = '24g'

Num_shuffle_partitions = input("spark.sql.shuffle.partitions:") or "288"

spark.executor.instances: 
spark.executor.cores: 
spark.executor.memory? Please end with 'g': 
spark.executor.memoryOverhead? Please end with 'g': 
spark.sql.shuffle.partitions: 


In [5]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
import geopandas as gpd
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType, ArrayType, MapType
from shapely.geometry import Point
from shapely.geometry import Polygon

from sedona.register import SedonaRegistrator
from sedona.core.SpatialRDD import SpatialRDD, PointRDD, CircleRDD, PolygonRDD, LineStringRDD
from sedona.core.enums import FileDataSplitter
from sedona.utils.adapter import Adapter
from sedona.core.spatialOperator import KNNQuery
from sedona.core.spatialOperator import JoinQuery
from sedona.core.spatialOperator import JoinQueryRaw
from sedona.core.spatialOperator import RangeQuery
from sedona.core.spatialOperator import RangeQueryRaw
from sedona.core.formatMapper.shapefileParser import ShapefileReader
from sedona.core.formatMapper import WkbReader
from sedona.core.formatMapper import WktReader
from sedona.core.formatMapper import GeoJsonReader
from sedona.sql.types import GeometryType
from sedona.core.enums import GridType
from sedona.core.SpatialRDD import RectangleRDD
from sedona.core.enums import IndexType
from sedona.core.geom.envelope import Envelope
from sedona.utils import SedonaKryoRegistrator, KryoSerializer

In [6]:
os.environ['PYSPARK_PYTHON'] = "./environment/bin/python"
#os.environ['PYSPARK_PYTHON'] = "/home/qiany/.conda/envs/py37/bin/python"
os.environ['YARN_CONF_DIR'] = "/opt/cloudera/parcels/CDH/lib/spark/conf/yarn-conf"

In [7]:
'''
spark.executor.cores: # Number of concurrent tasks an executor can run, euqals to the number of cores to use on each executor
spark.executor.instances: # Number of executors for the spark application
spark.executor.memory: # Amount of memory to use for each executor that runs the task
spark.executor.memoryOverhead:
spark.driver.cores: # Number of cores to use for the driver process; the default number is 1
spark.driver.memory: # Amount of memory to use for the driver
spark.driver.maxResultSize: to define the maximum limit of the total size of the serialized result that a driver can store for each Spark collect action
spark.default.parallelism: # Default number of partitions in RDDs returned by transformations like join, reduceByKey, and parallelize when not set by user. It can be set as spark.executor.instances * spark.executor.cores * 2
spark.sql.shuffle.partitions: determine how many partitions are used when data is shuffled between nodes, e.g., joins or aggregations. usually 1~5 times of executor.instances * executor.cores
spark.memory.storageFraction: determines the fraction of the heap space that is allocated to caching RDDs and DataFrames in memory.
spark.kryoserializer.buffer.max: determine the maximum of data that can be serialized at once; this must be larger than any object we attempt to serialize
spark.rpc.message.maxSize: # Maximum message size (in MiB) to allow in "control plane" communication; generally only applies to map output size information sent between executors and the driver. To communicate between the nodes, Spark uses a protocol called RPC (Remote Procedure Call), which sends messages back and forth. The spark.rpc.message.maxSize parameter limits how big these messages can be. 
spark.sql.broadcastTimeout: Spark will wait for this amount of time before giving up on broadcasting a table. Broadcasting can take a long time if the table is large or if there is a shuffle operation before it.
spark.sql.autoBroadcastJoinThreshold: Spark will broadcast a table to all worker nodes when performing a join if its size is less than this value; -1 means disabling broadcasting
'''

date = time.strftime("%m,%d,%Y")
date_name = date.split(',')[0] + date.split(',')[1] + date.split(',')[2]

hour = time.strftime("%H,%M")
hour_name = hour.split(',')[0] + hour.split(',')[1]

# set the Spark app name
spark_app_name = "Seaice_step9_resimplify_after_merge_r" + round_idx + "_step6_visualize_" + tin_filename + '_' + date_name + '_' + hour_name
print("spark_app_name:", spark_app_name)

spark = SparkSession \
.builder \
.appName(spark_app_name) \
.master('yarn') \
.config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
.config("spark.kryo.registrator", SedonaKryoRegistrator.getName) \
.config('spark.jars','sedona-core-2.4_2.11-1.0.0-incubating.jar,sedona-sql-2.4_2.11-1.0.0-incubating.jar,sedona-python-adapter-2.4_2.11-1.0.0-incubating.jar,sedona-viz-2.4_2.11-1.0.0-incubating.jar,geotools-wrapper-geotools-24.0.jar,graphframes-0.8.0-spark2.4-s_2.11.jar') \
.config('spark.executor.cores', Num_core_per_executor) \
.config('spark.executor.instances', Num_executor) \
.config('spark.executor.memory', Memory_executor) \
.config('spark.executor.memoryOverhead', MemoryOverhead_executor) \
.config('spark.driver.cores', Num_core_per_driver) \
.config('spark.driver.memory', Memory_driver) \
.config('spark.driver.memoryOverhead', MemoryOverhead_driver) \
.config('spark.driver.maxResultSize', '0') \
.config('spark.dynamicAllocation.enabled', 'false') \
.config('spark.network.timeout', '10000001s') \
.config('spark.executor.heartbeatInterval', '10000000s') \
.config('spark.sql.shuffle.partitions', Num_shuffle_partitions) \
.config("spark.default.parallelism", '200') \
.config("spark.kryoserializer.buffer.max", "1024mb") \
.config('spark.rpc.message.maxSize', '256') \
.config("spark.sql.broadcastTimeout", "36000") \
.config("spark.sql.autoBroadcastJoinThreshold", "-1") \
.config("spark.python.profile", "true") \
.config("spark.eventLog.enabled", "true") \
.config('spark.yarn.dist.archives', '/local/data/yuehui/py37.tar.gz#environment') \
.getOrCreate()

spark_app_name: Seaice_step9_resimplify_after_merge_r6_step6_visualize_ALS_L1B_20190410T153919_155000_1_04282026_1402


In [8]:
import sys
sys.path.append("/local/data/yuehui/pyspark/Topo_Simplification/code")

In [9]:
import graphframes
from graphframes import GraphFrame
from graphframes import *
from graphframes.lib import Pregel

In [10]:
file_df_ver_order = directory + '/' + tin_filename + '_filtra_pts' + '.parquet'
df_ver_order = spark.read.parquet(file_df_ver_order)
df_ver_order.printSchema()

root
 |-- x: float (nullable = true)
 |-- y: float (nullable = true)
 |-- ele: float (nullable = true)
 |-- self_index: integer (nullable = true)
 |-- self_order: integer (nullable = true)



#### get df_crit_E_tri

In [11]:
file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '.parquet'

# file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_from_r1_merge_r2_resimplify_r1' + '.parquet'
df_crit_E_Co_tris = spark.read.parquet(file_df_crit_E_Co_tris)
df_crit_E_Co_tris.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_Co_t: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [12]:
df_crit_E_tri = df_crit_E_Co_tris.groupby("Saddle_edge").agg(collect_list('Saddle_Co_t').alias('saddle_edge_Tri'))
df_crit_E_tri.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- saddle_edge_Tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [13]:
file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '.parquet'

# file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_from_r1_merge_r2_resimplify_r1' + '.parquet'
df_V1_result_VE_update_final = spark.read.parquet(file_df_V1_result_VE_update_final)
df_V1_result_VE_update_final.printSchema()

root
 |-- component: long (nullable = true)
 |-- final_V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [14]:
file_df_crit_V_update = directory + '/' + tin_filename + '_df_crit_V_update_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '.parquet'

# file_df_crit_V_update = directory + '/' + tin_filename + '_df_crit_V_update_from_r1_merge_r2_resimplify_r1' + '.parquet'
df_crit_V_update = spark.read.parquet(file_df_crit_V_update)
df_crit_V_update.printSchema()

root
 |-- Ver: integer (nullable = true)
 |-- Min_ver: integer (nullable = true)



In [15]:
file_df_crit_E_update = directory + '/' + tin_filename + '_df_crit_E_update_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '.parquet'

# file_df_crit_E_update = directory + '/' + tin_filename + '_df_crit_E_update_from_r1_merge_r2_resimplify_r1' + '.parquet'
df_crit_E_update = spark.read.parquet(file_df_crit_E_update)
df_crit_E_update.printSchema()

root
 |-- Ver: integer (nullable = true)
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [16]:
file_df_crit_T_update = directory + '/' + tin_filename + '_df_crit_T_update_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '.parquet'

# file_df_crit_T_update = directory + '/' + tin_filename + '_df_crit_T_update_from_r1_merge_r2_resimplify_r1' + '.parquet'
df_crit_T_update = spark.read.parquet(file_df_crit_T_update)
df_crit_T_update.printSchema()

root
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [17]:
file_DF_saddle_minima_remove = directory + '/' + tin_filename + '_DF_saddle_minima_remove_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '.parquet'

# file_DF_saddle_minima_remove = directory + '/' + tin_filename + '_DF_saddle_minima_remove_from_r1_merge_r2_resimplify_r1' + '.parquet'
DF_saddle_minima_remove = spark.read.parquet(file_DF_saddle_minima_remove)
DF_saddle_minima_remove.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- removed_minimum: long (nullable = true)



In [18]:
file_DF_maxima_saddle_remove = directory + '/' + tin_filename + '_DF_maxima_saddle_remove_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '.parquet'

# file_DF_maxima_saddle_remove = directory + '/' + tin_filename + '_DF_maxima_saddle_remove_from_r1_merge_r2_resimplify_r1' + '.parquet'
DF_maxima_saddle_remove = spark.read.parquet(file_DF_maxima_saddle_remove)
DF_maxima_saddle_remove.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- removed_crti_triangle: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [19]:
DF_saddles_remove_VE = DF_saddle_minima_remove.select("removed_saddle")
DF_saddles_remove_ET = DF_maxima_saddle_remove.select("removed_saddle")

DF_saddles_remove = DF_saddles_remove_VE.union(DF_saddles_remove_ET)
DF_saddles_remove.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### sort df_ver_order according to the self_order value, which is the filtration value

In [20]:
df_ver_order = df_ver_order.sort(col('self_order'), ascending=True)

# collect df_ver_order as a global array 
df_ver_order_col = df_ver_order.collect()

### save ascending 1-manifolds, the V2-paths

In [21]:
# df_graph_V2_node_final contains a special node -1
file_df_graph_V2_node_final = directory + '/' + tin_filename + '_ET_node' + '.parquet'
df_graph_V2_node_final = spark.read.parquet(file_df_graph_V2_node_final)
df_graph_V2_node_final.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [22]:
file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '.parquet'

# file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_from_r1_merge_r2_resimplify_r1' + '.parquet'
result_con_ET = spark.read.parquet(file_result_con_ET)
result_con_ET.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)



##### collect the V2-paths as a global array

In [23]:
file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_eliminate_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '.parquet'

# file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_eliminate_from_r1_merge_r2_resimplify_r1' + '.parquet'
df_V2_result_ET_update_final = spark.read.parquet(file_df_V2_result_ET_update_final)
df_V2_result_ET_update_final.printSchema()

root
 |-- component: long (nullable = true)
 |-- final_V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [24]:
df_V2_result_ET_update_final_expl = df_V2_result_ET_update_final.select(explode(df_V2_result_ET_update_final.final_V2_paths).alias("V2_path"))
df_V2_result_ET_update_final_expl.printSchema()

t_0 = time.time()

df_V2_result_ET_update_final_expl_col = df_V2_result_ET_update_final_expl.collect()

t_1 = time.time()
print("Time cost to collect V1-paths of Graph_ET:", t_1 - t_0)

root
 |-- V2_path: array (nullable = true)
 |    |-- element: integer (containsNull = true)

Time cost to collect V1-paths of Graph_ET: 3.7479100227355957


##### sort df_graph_V2_node according to the id, df_graph_V2_node does not contain node -1

In [25]:
df_graph_V2_node = df_graph_V2_node_final.filter(df_graph_V2_node_final.id != -1)
df_graph_V2_node = df_graph_V2_node.sort(col('id'), ascending=True)

# collect df_graph_V2_node_final as a global array 
df_graph_V2_node_col = df_graph_V2_node.collect()

##### get the triangle id of critical triangle, whose triangle id is not equal to the component id

In [26]:
# given the triangle id, obtain the xyz of the baricenter of this triangle
def baricenterXYZ(tri_id):
    tri_end_pts = df_graph_V2_node_col[tri_id]['tri']
    v_0 = tri_end_pts[0]
    v_1 = tri_end_pts[1]
    v_2 = tri_end_pts[2]
    x = (df_ver_order_col[v_0]['x'] + df_ver_order_col[v_1]['x'] + df_ver_order_col[v_2]['x'])/3.0
    y = (df_ver_order_col[v_0]['y'] + df_ver_order_col[v_1]['y'] + df_ver_order_col[v_2]['y'])/3.0
    z = (df_ver_order_col[v_0]['ele'] + df_ver_order_col[v_1]['ele'] + df_ver_order_col[v_2]['ele'])/3.0
    return x, y, z

In [27]:
# given the triangle id, obtain the xyz of the baricenter of this triangle
def baricenterXYZ_VerArray(v_0, v_1, v_2):
    x = (df_ver_order_col[v_0]['x'] + df_ver_order_col[v_1]['x'] + df_ver_order_col[v_2]['x'])/3.0
    y = (df_ver_order_col[v_0]['y'] + df_ver_order_col[v_1]['y'] + df_ver_order_col[v_2]['y'])/3.0
    z = (df_ver_order_col[v_0]['ele'] + df_ver_order_col[v_1]['ele'] + df_ver_order_col[v_2]['ele'])/3.0
    return x, y, z

In [28]:
# given teh triangle ids of two adjacent triangle, obtain the xyz of the mid-point of the common edge
def mid_pt_XYZ_edge(v_0, v_1):
    x = (df_ver_order_col[v_0]['x'] + df_ver_order_col[v_1]['x'])/2.0
    y = (df_ver_order_col[v_0]['y'] + df_ver_order_col[v_1]['y'])/2.0
    z = (df_ver_order_col[v_0]['ele'] + df_ver_order_col[v_1]['ele'])/2.0
    return x, y, z

In [29]:
# given teh triangle ids of two adjacent triangle, obtain the xyz of the mid-point of the common edge
def mid_pt_XYZ(tri_id_0, tri_id_1):
    tri_end_pts_0 = df_graph_V2_node_col[tri_id_0]['tri']
    tri_end_pts_1 = df_graph_V2_node_col[tri_id_1]['tri']
    # get the common edge of these two triangles
    intersection = set(tri_end_pts_0) & set(tri_end_pts_1)
    edge = list(intersection)
    
    v_0 = edge[0]
    v_1 = edge[1]
    x = (df_ver_order_col[v_0]['x'] + df_ver_order_col[v_1]['x'])/2.0
    y = (df_ver_order_col[v_0]['y'] + df_ver_order_col[v_1]['y'])/2.0
    z = (df_ver_order_col[v_0]['ele'] + df_ver_order_col[v_1]['ele'])/2.0
    return x, y, z

In [30]:
# the saddles to be visualized include two parts: the final critical edges, the saddles that are removed when contracting saddles-maxima
df_crit_E_update_saddle = df_crit_E_update.select("Saddle_edge")
df_crit_E_to_be_write_A1 = df_crit_E_update_saddle.union(DF_saddles_remove_ET)

df_crit_E_to_be_write_A1.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [31]:
# the co-boundary triangles of saddles to be write
df_crit_E_to_be_write_A1_co_t = df_crit_E_to_be_write_A1.join(df_crit_E_tri, df_crit_E_to_be_write_A1.Saddle_edge==df_crit_E_tri.Saddle_edge, "left")
df_crit_E_to_be_write_A1_co_t = df_crit_E_to_be_write_A1_co_t.select(df_crit_E_tri.Saddle_edge, df_crit_E_tri.saddle_edge_Tri)
df_crit_E_to_be_write_A1_co_t.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- saddle_edge_Tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



#### filter according to HA by considering only critical triangles

In [34]:
HA_threshold = 0.5

In [35]:
# given the triangle id, obtain the minimum elevation of its boundary vertices
def tri_HA(tri_id):
    tri_end_pts = df_graph_V2_node_col[tri_id]['tri']
    v_0 = tri_end_pts[0]
    v_1 = tri_end_pts[1]
    v_2 = tri_end_pts[2]
    ele_mini = min(df_ver_order_col[v_0]['ele'], df_ver_order_col[v_1]['ele'], df_ver_order_col[v_2]['ele'])
    
    return ele_mini

In [36]:
# given the array of boundary vertices of a triangle, obtain the minimum elevation of its boundary vertices
def tri_ary_HA(tri_ary):
    v_0 = tri_ary[0]
    v_1 = tri_ary[1]
    v_2 = tri_ary[2]
    ele_mini = min(df_ver_order_col[v_0]['ele'], df_ver_order_col[v_1]['ele'], df_ver_order_col[v_2]['ele'])
    
    return ele_mini

#### filter according to roughness by considering all individual triangles

In [37]:
roughness_threshold = 0.02

In [38]:
file_df_VV_roughness = directory + '/' + tin_filename + '_roughness_VV_2ord' + '.parquet'
df_VV_roughness = spark.read.parquet(file_df_VV_roughness)
df_VV_roughness.printSchema()

df_VV_roughness = df_VV_roughness.sort(col('Ver'), ascending=True)

# collect df_roughness as a global array 
df_VV_roughness_col = df_VV_roughness.collect()

root
 |-- Ver: integer (nullable = true)
 |-- roughness: double (nullable = true)



In [39]:
# given the triangle id, obtain the minimum elevation of its boundary vertices
def tri_roughness(tri_id):
    tri_end_pts = df_graph_V2_node_col[tri_id]['tri']
    v_0 = tri_end_pts[0]
    v_1 = tri_end_pts[1]
    v_2 = tri_end_pts[2]
    roughness_mini = min(df_VV_roughness_col[v_0]['roughness'], df_VV_roughness_col[v_1]['roughness'], df_VV_roughness_col[v_2]['roughness'])
    
    return roughness_mini

In [40]:
# given the array of boundary vertices of a triangle, obtain the minimum elevation of its boundary vertices
def tri_ary_roughness(tri_ary):
    v_0 = tri_ary[0]
    v_1 = tri_ary[1]
    v_2 = tri_ary[2]
    roughness_mini = min(df_VV_roughness_col[v_0]['roughness'], df_VV_roughness_col[v_1]['roughness'], df_VV_roughness_col[v_2]['roughness'])
    
    return roughness_mini

#### soft_HA, do not remove short ridges

In [41]:
def write_A1_V2paths_soft_HA_roughness_csv(output_vtk, output_csv, df_crit_E_tri_col, df_V2_result_ET_init_col):
    vertices_num = 0
    edges_num = 0
    num_rmShortRidge = 0
    for i in range(0, len(df_V2_result_ET_init_col)):
        vertices_temp = 0
        edges_temp = 0
        ridge_len_temp = 0
        if df_V2_result_ET_init_col[i] and df_V2_result_ET_init_col[i]['V2_path'] and df_V2_result_ET_init_col[i]['V2_path'][-1] != -1: # this is a valid V2-path
            tri_index = df_V2_result_ET_init_col[i]['V2_path'][-1] # df_V2_result_ET_init_col[i]['V2_path'][-1] is a critical triangle
            if tri_HA(tri_index) < HA_threshold or tri_roughness(tri_index) < roughness_threshold:
                continue
                
            # first for loop to get the length
            for j in range(len(df_V2_result_ET_init_col[i]['V2_path'])-1, -1, -1):
                tri_index = df_V2_result_ET_init_col[i]['V2_path'][j]
                if tri_roughness(tri_index) >= roughness_threshold:
                    ridge_len_temp += 1
                    vertices_temp += 1
                else:
                    break                    
                            
            if vertices_temp > 0:
                actual_pts_num = vertices_temp*2 - 1
                vertices_num = vertices_num + actual_pts_num
                edges_num = edges_num + actual_pts_num - 1
    
    with open(output_csv, "w", newline="") as csv_f:
        csv_writer = csv.writer(csv_f)
        csv_writer.writerow(["eid", "geometry", "first vertex elevation", "second vertex elevation"])
        eid_csv = 1  # edge id in CSV
        
        # We write points in the same order as that in the VTK file
        # For CSV, we output a segment each time we have two consecutive points in a ridge polyline.
        def write_segment_to_csv(p0, p1, eid_csv):
            x0, y0, z0 = p0
            x1, y1, z1 = p1
            wkt = f'LINESTRING ({x0:.15f} {y0:.15f}, {x1:.15f} {y1:.15f})'
            csv_writer.writerow([eid_csv, wkt, f"{z0:.15f}", f"{z1:.15f}"])
        
        # write vertices along V2-paths
        num_1 = 0
        for i in range(0, len(df_V2_result_ET_init_col)):
            write_last_triangle = False
            ridge_len_temp = 0
            if df_V2_result_ET_init_col[i] and df_V2_result_ET_init_col[i]['V2_path'] and df_V2_result_ET_init_col[i]['V2_path'][-1] != -1: # this is a valid V2-path
                tri_index = df_V2_result_ET_init_col[i]['V2_path'][-1] # df_V2_result_ET_init_col[i]['V2_path'][-1] is a critical triangle
                if tri_HA(tri_index) < HA_threshold or tri_roughness(tri_index) < roughness_threshold:
                    continue                      
                                    
                prev_pt = None
                for j in range(len(df_V2_result_ET_init_col[i]['V2_path'])-1, 0, -1):
                    tri_index = df_V2_result_ET_init_col[i]['V2_path'][j]
                    tri_index_prev = df_V2_result_ET_init_col[i]['V2_path'][j-1]
                    if tri_roughness(tri_index) >= roughness_threshold and tri_roughness(tri_index_prev) >= roughness_threshold:
                        # get baricenter of current triangle
                        tri_index = df_V2_result_ET_init_col[i]['V2_path'][j]
                        tri_bari = baricenterXYZ(tri_index) # (x, y, z)
                        if prev_pt is not None:
                            write_segment_to_csv(prev_pt, tri_bari, eid_csv)
                            eid_csv += 1
                        prev_pt = tri_bari
                        num_1 += 1
                        
                        # get mid-point of common edge of current triangle and the previous triangle
                        tri_index_prev = df_V2_result_ET_init_col[i]['V2_path'][j-1]
                        mid_pt = mid_pt_XYZ(tri_index, tri_index_prev)
                        write_segment_to_csv(prev_pt, mid_pt, eid_csv)
                        eid_csv += 1
                        prev_pt = mid_pt
                        num_1 += 1
                    elif tri_roughness(tri_index) >= roughness_threshold:
                        # get baricenter of current triangle, which is current triangle
                        tri_index = df_V2_result_ET_init_col[i]['V2_path'][j]
                        tri_bari = baricenterXYZ(tri_index) # (x, y, z)
                        num_1 += 1
                        
                        if prev_pt is not None:
                            write_segment_to_csv(prev_pt, tri_bari, eid_csv)
                            eid_csv += 1
                        prev_pt = tri_bari
                        
                        write_last_triangle = True
                    else:
                        break
                # get the baricenter of the last triangle
                if write_last_triangle == False:
                    tri_index = df_V2_result_ET_init_col[i]['V2_path'][0]
                    tri_bari_last = baricenterXYZ(tri_index)
                    if prev_pt is not None:
                        write_segment_to_csv(prev_pt, tri_bari_last, eid_csv)
                        eid_csv += 1
                    num_1 += 1                    
                
    print("Writing ascending 1 cell complete.")
    print("eid_csv:", eid_csv)

In [42]:
t_0 = time.time()

output_vtk_with_saddle = '/local/data/yuehui/pyspark/data/Part3_TopoSim/Part3_visualization/' + directory + '/' + tin_filename + '_A1_toposimp' + '_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '_simplify_ele_' + porsist_thre + '_soft_HA_' + str(HA_threshold) + '_rough_' + str(roughness_threshold) + '_' + date_name + '.vtk'
output_csv_with_saddle = '/local/data/yuehui/pyspark/data/Part3_TopoSim/Part3_visualization/' + directory + '/' + tin_filename + '_A1_toposimp' + '_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '_simplify_ele_' + porsist_thre + '_soft_HA_' + str(HA_threshold) + '_rough_' + str(roughness_threshold) + '_' + date_name + '.csv'

# output_vtk_with_saddle = '/local/data/yuehui/pyspark/data/Part3_TopoSim/Part3_visualization/' + directory + '/' + tin_filename + '_A1_saddle_toposimp_' + date_name + '_from_r' + already_resimplify_idx + '_merge_r' + already_merge_idx + '_resimplify_r' + round_idx + '_simplify_ele_0.25_HA_' + str(HA_threshold) + '_rough_' + str(roughness_threshold) + '.vtk'

write_A1_V2paths_soft_HA_roughness_csv(output_vtk_with_saddle, output_csv_with_saddle, df_crit_E_to_be_write_A1_co_t_col, df_V2_result_ET_update_final_expl_col)

t_1 = time.time()
print(f"write to {output_csv_with_saddle}")

print("Time cost to write_A1_V2paths:", t_1 - t_0)

Writing ascending 1 cell complete.
eid_csv: 2822617
write to /local/data/yuehui/pyspark/data/Part3_TopoSim/Part3_visualization/ALS_L1B_20190410T153919_155000_1_pers_020/ALS_L1B_20190410T153919_155000_1_A1_toposimp_from_r1_merge_r2_resimplify_r6_simplify_ele_0.20_soft_HA_0.67_rough_0.02_04282026.csv
Time cost to write_A1_V2paths: 68.04250049591064


In [43]:
t_whole_program_1 = time.time()
print(f"Total time cost (min) of the whole program: {(t_whole_program_1 - t_whole_program_0)/60}")

Total time cost (min) of the whole program: 7.958261815706889
